In order to convert the acceleration data (AD) given in bits into gravity, use this equation: 

Acceleration [g]: [(2*Range)/(2^Resolution)]*AD

In order to convert the rotation data (RD) given in bits into angular velocity, use this equation:

Angular velocity [°/s]: [(2*Range)/(2^Resolution)]*RD


Importamos las librerias.


In [36]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define la ruta de un archivo (asegúrate de apuntar a tu carpeta de datos)
ruta_archivo = "Archivos caídas/F03_SA03_R03.txt"

# 2. Abre el archivo, elimina los ';' al final de cada fila y carga los números
with open(ruta_archivo, "r") as f:
  lineas_limpias = [linea.replace(";", "") for linea in f]

datos_crudos = np.loadtxt(lineas_limpias, delimiter=",")

# 3. Verificación de dimensiones
print("Forma de la matriz (filas, columnas):", datos_crudos.shape)
print("Primera fila de datos crudos:\n", datos_crudos[0])

Forma de la matriz (filas, columnas): (3000, 9)
Primera fila de datos crudos:
 [   4. -257.  -73.  -39.   71.    6.  -57. -988. -138.]


Las 3 primeras colummnas de los datos crudos, corresponden a los ejes X,Y,Z del acelerometro ADXL345, el cual entrega los datos en bits, con una resolucion de 13 bits y un rango de $\pm 16g$ 
para convertir los datos de aceleracion a unidades de gravedad, usamos la ecuacion: 
Acceleration [g]: [(2*Range)/(2^Resolution)]*AD, dada en el archivo README

Debemos aplicar una norma vectorial a los datos para obtener su magnitud, asi obtenemos una señal unica que no depende de la orientacion del sensor
$$a_{\text{total}}(t) = \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2}$$

In [37]:
import plotly.graph_objects as go

# 1. Factor de conversión del acelerómetro ADXL345 (según Readme: +-16g, 13 bits)
factor_g = (2 * 16) / (2**13)

# 2. Extraer columnas 0, 1 y 2 (X, Y, Z) y multiplicar por el factor
ax = datos_crudos[:, 0] * factor_g
ay = datos_crudos[:, 1] * factor_g
az = datos_crudos[:, 2] * factor_g

# 3. Calcular la magnitud de la aceleración total en cada instante t
a_total = np.sqrt(ax**2 + ay**2 + az**2)

# 4. Crear el vector de tiempo en segundos (frecuencia de muestreo = 200 Hz)

#Debido a que los archivos no nos dan los datos de los segundos, tenemos que construir el eje de tiempo a partir 
#de la cantidad de muestras y la frecuencia de muestreo. 
#La frecuencia de muestreo es de 200 Hz, es decir, 200 muestras por segundo. 
#Por lo tanto, el tiempo en segundos se puede calcular como el número de muestras dividido por la frecuencia de muestreo.
fs_muestreo = 200
tiempo = np.arange(len(a_total)) / fs_muestreo

# 5. Verificación visual en el dominio del tiempo
fig = go.Figure()
fig.add_trace(go.Scatter(x=tiempo, y=a_total, mode="lines"))
fig.update_layout(
    title="Aceleración Total a_total(t)",
    xaxis_title="Tiempo (s)",
    yaxis_title="Aceleración (g)",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()

Ahora necesitamos eliminar la señal constante de la gravedad:
Para eso, primero calculamos el promedio aritmetico de toda la señal calculando un promedio de esta usando la funcion np.mean()
luego en cada muestra individual de a_total, es decir, las magnitudes en diferentes tiempo de la señal, se le resta el valor promedio hallado, que seria aproximadamente 1g correspondiente a la gravedad.

la variable N, guarda la cantidad total de emuestras de la señal (por ejemplo 3000 muestras para un archivo de 15s)

np.fft.rfft(a_sin_dc): Ejecuta el algoritmo de la Transformada Rápida de Fourier para entradas reales (Real Fast Fourier Transform) y guarda el resultado en fft_compleja

np.abs(fft_compleja) / N: Calcula el módulo de cada número complejo resultante y lo divide entre el número total de muestras N. El módulo corresponde al espectro de magnitud $\vert{}F(\omega)\vert{}$ de la Transformada de Fourier. Dividir entre N normaliza la amplitud respecto a la duración del registro, permitiendo comparar en la misma escala archivos cortos de caídas ($15\text{ s}$, $N=3000$) frente a archivos largos de actividades diarias ($100\text{ s}$, $N=20000$).


In [38]:
import plotly.graph_objects as go

# 1. Quitar la componente constante de gravedad (0 Hz / DC)
a_sin_dc = a_total - np.mean(a_total)

# 2. Calcular la FFT para señales reales y su magnitud normalizada
N = len(a_sin_dc)
fft_compleja = np.fft.rfft(a_sin_dc)
magnitud = np.abs(fft_compleja) / N

# 3. Construir el eje X de frecuencias en Hz (sabemos que fs_muestreo = 200 Hz)
frecuencias = np.fft.rfftfreq(N, d=1 / fs_muestreo)

# 4. Graficar el espectro de frecuencias
fig = go.Figure()
fig.add_trace(go.Scatter(x=frecuencias, y=magnitud, mode="lines"))
fig.update_layout(
    title="Espectro de Frecuencias |F(f)| - Caída F03",
    xaxis_title="Frecuencia (Hz)",
    yaxis_title="Magnitud",
    xaxis=dict(range=[0, 50], showgrid=True),  # Hacemos zoom de 0 a 50 Hz donde ocurre el movimiento humano
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()

Este bloque de codigo, lo que hace en resumen, es volver a aplicar los codigos anteriores, pero a los archivos especificados en el diccionario, superponerlos mediante un plot y compararlos para obtener los fs y fp.
Físicamente, el impacto de una caída contra el suelo se aproxima a un impulso abrupto, por lo que proyecta densidad espectral $\vert{}F(\omega)\vert{}$ hacia frecuencias altas. Por el contrario, las actividades cotidianas como caminar y trotar son movimientos rítmicos/periódicos cuya energía se concentra en frecuencias bajas.

El ciclo for extrae cada pareja clave-valor del diccionario. En cada iteración invoca a obtener_espectro(ruta) y trazala curva mediante plt.plot(..., alpha=0.8), donde alpha controla la opacidad ($80\%$) para que las líneas superpuestasno se oculten entre sí.


En la teoría de Fourier, una señal periódica o repetitiva en el tiempo concentra toda su energía en frecuencias discretas muy estrechas (la frecuencia fundamental y sus armónicos). Trotar (D04) es un movimiento rítmico de 100 segundos donde la persona da aproximadamente 3 zancadas por segundo de manera constante; por eso toda la energía se apila en una torre angosta en $f \approx 3.0\text{ Hz}$ y sus múltiplos ($5.8\text{ Hz}$, $8.7\text{ Hz}$).   Por el contrario, una caída (F05, F06) es un evento aperiódico y abrupto (un impacto de fracción de segundo dentro de un registro de $15\text{ s}$). Por propiedades de la Transformada de Fourier, un pulso corto y brusco en el tiempo esparce su energía a lo largo de un ancho de banda muy amplio (desde $0\text{ Hz}$ hasta casi $30\text{ Hz}$), en lugar de concentrarla en un solo pico alto

El filtro a escoger seria **pasa altas**, debido a que las actividades presentan picos en las frecuencias bajas y disminuyen de forma mucho más abrupta en frecuencias altas, sin embargo, actividades anomalas como las caidas, las cuales no presentan un ritmo, mantiene su frecuencia mejor a lo largo del ancho de banda.

Para la frecuencia de rechazo fs, escogemos un valor de 10Hz, ya que las actividades de trotar y caminar presentan picos por debajo de dicha frecuencia

Para la frecuencia de paso, escogemos 16Hz, ya que a partir de dicha frecuencia, el trote o el caminar ya no mantiene su energia en comparacion con las actividades anomalas como las caidas

In [39]:
import numpy as np
import plotly.graph_objects as go


# 1. Función reutilizable que recibe la ruta de UN archivo y devuelve su FFT
def obtener_espectro(ruta):
    # Leer y limpiar los ';'
    with open(ruta, "r") as f:
        lineas = [linea.replace(";", "") for linea in f]
    datos = np.loadtxt(lineas, delimiter=",")

    # Convertir a g (ADXL345: 13 bits, +-16g) y calcular magnitud total[cite: 1, 2]
    factor_g = (2 * 16) / (2**13)
    ax = datos[:, 0] * factor_g
    ay = datos[:, 1] * factor_g
    az = datos[:, 2] * factor_g
    a_total = np.sqrt(ax**2 + ay**2 + az**2)

    # Quitar gravedad constante (0 Hz) y calcular FFT normalizada
    a_sin_dc = a_total - np.mean(a_total)
    N = len(a_sin_dc)
    magnitud = np.abs(np.fft.rfft(a_sin_dc)) / N
    frecuencias = np.fft.rfftfreq(N, d=1 / 200)

    return frecuencias, magnitud


# 2. Diccionario con los 4 archivos clave de la Fase 1 (ajusta las rutas a tus carpetas)
archivos = {
    "Caminar lento (D01)": r"Archivos caídas/D01_SA01_R01.txt",
    "Trotar rápido (D04)": r"Archivos caídas/D04_SA03_R01.txt",
    "Caída 1 (F05)": r"Archivos caídas/F05_SA03_R01.txt",
    "Caída 2 (F06)": r"Archivos caídas/F06_SA03_R01.txt",
}

# 3. Crear la figura interactiva en Plotly y agregar las 4 curvas superpuestas
fig = go.Figure()

for etiqueta, ruta in archivos.items():
    freqs, mag = obtener_espectro(ruta)
    fig.add_trace(
        go.Scatter(
            x=freqs,
            y=mag,
            mode="lines",
            name=etiqueta,
            opacity=0.8,
        )
    )

# 4. Configurar títulos, ejes y zoom entre 0 y 40 Hz
fig.update_layout(
    title="Comparación Espectral: Actividades Diarias vs. Caídas",
    xaxis_title="Frecuencia (Hz)",
    yaxis_title="Magnitud |F(f)|",
    xaxis=dict(range=[0, 40], showgrid=True),
    yaxis=dict(showgrid=True),
    template="plotly_white",
    width=1000,
    height=450,
    hovermode="x unified",
)

fig.show()

### Importación de archivos como dataframes

In [40]:
import pandas as pd
from pathlib import Path

# Carpeta con los .txt (relativa a la carpeta del notebook/script: Proyecto_SisFall)
CARPETA = Path("Archivos caídas")

COLUMNAS = [
    'ADXL_X', 'ADXL_Y', 'ADXL_Z',  # Acelerómetro 1
    'ITG_X', 'ITG_Y', 'ITG_Z',     # Giroscopio
    'MMA_X', 'MMA_Y', 'MMA_Z'      # Acelerómetro 2
]

# Factores de conversión: (2 * Rango) / (2^Resolución)
FACTORES = {
    'ADXL': (2 * 16) / 2**13,    # ±16 g, 13 bits
    'ITG':  (2 * 2000) / 2**16,  # ±2000 °/s, 16 bits
    'MMA':  (2 * 8) / 2**14,     # ±8 g, 14 bits
}

def importar_y_limpiar(nombre_archivo, carpeta=CARPETA, magnitud=True):
    """
    Importa un .txt de SisFall, lo limpia y lo convierte a unidades físicas.
    - nombre_archivo: con o sin extensión, ej. 'D04_SA01_R01' o 'D04_SA01_R01.txt'
    - magnitud: True  -> devuelve 3 columnas: ADXL, ITG, MMA (magnitud del vector X,Y,Z)
                False -> devuelve las 9 columnas por componente (ADXL_X, ADXL_Y, ...)
    - Devuelve el DataFrame y además lo deja en memoria como df_<nombre>.
    """
    nombre = Path(nombre_archivo).stem
    ruta = Path(carpeta) / f"{nombre}.txt"

    # 1. Lectura
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)

    # 2. Limpieza (el último dato de cada fila trae ';')
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)

    # 3. Conversión de bits a valores físicos
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor

    # 4. Magnitud del vector resultante (opcional)
    if magnitud:
        df = pd.DataFrame({
            prefijo: np.sqrt(df[f'{prefijo}_X']**2 +
                             df[f'{prefijo}_Y']**2 +
                             df[f'{prefijo}_Z']**2)
            for prefijo in FACTORES
        })

    # 5. Guardar en memoria con el nombre df_<archivo>
    globals()[f"df_{nombre}"] = df
    print(f"✔ '{nombre}.txt' cargado → df_{nombre} ({df.shape[0]} filas, columnas: {list(df.columns)})")
    return df

    # 4. Guardar en memoria con el nombre df_<archivo>
    globals()[f"df_{nombre}"] = df
    print(f"✔ '{nombre}.txt' cargado → df_{nombre} ({df.shape[0]} filas)")
    return df

In [41]:
# Ejemplo de uso:

df_D04_SA01_R01 = importar_y_limpiar("D04_SA01_R01")
df_D04_SA01_R01.head()

✔ 'D04_SA01_R01.txt' cargado → df_D04_SA01_R01 (20000 filas, columnas: ['ADXL', 'ITG', 'MMA'])


,ADXL,ITG,MMA
0,1.698936,70.107266,1.649967
1,1.677401,69.255756,1.629241
2,1.702879,73.634053,1.654461
3,1.698689,75.222927,1.634243
4,1.691956,73.902914,1.636463


### Función Graficar en Tiempo

In [42]:
import numpy as np
import plotly.graph_objects as go

FS = 200  # Frecuencia de muestreo (Hz)


def graficar_serie_tiempo(df, columna, actividad=None, fs=FS, color='royalblue', devolver=False):
    if columna not in df.columns:
        raise ValueError(f"'{columna}' no existe. Columnas disponibles: {list(df.columns)}")

    tiempo = np.arange(len(df)) / fs
    unidad_y = "Velocidad Angular (°/s)" if columna.startswith("ITG") else "Aceleración (g)"

    titulo = f"Serie de Tiempo: Sensor {columna}"
    if actividad:
        titulo += f" - {actividad}"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=tiempo,
        y=df[columna],
        mode='lines',
        name=columna,
        line=dict(width=2, color=color)
    ))

    fig.update_layout(
        title=titulo,
        xaxis_title="Tiempo (Segundos)",
        yaxis_title=unidad_y,
        template="plotly_white",
        hovermode="x unified"
    )

    fig.show()
    if devolver:
        return fig

In [43]:
df_D04_SA01_R01_XYZ = importar_y_limpiar("D04_SA01_R01", magnitud=False)  # Cambiar a True si se desea la magnitud
graficar_serie_tiempo(df_D04_SA01_R01_XYZ, 'ADXL_Y', actividad="Persona trotando suave")

✔ 'D04_SA01_R01.txt' cargado → df_D04_SA01_R01 (20000 filas, columnas: ['ADXL_X', 'ADXL_Y', 'ADXL_Z', 'ITG_X', 'ITG_Y', 'ITG_Z', 'MMA_X', 'MMA_Y', 'MMA_Z'])


##### Caída al Caminar Lento en el Tiempo

In [44]:
df_D01_SA01_R01_mag =importar_y_limpiar("D01_SA01_R01", magnitud=True)  # Cambiar a False si no se desea la magnitud
graficar_serie_tiempo(df_D01_SA01_R01_mag, 'ADXL', actividad="Caminando Lento")

✔ 'D01_SA01_R01.txt' cargado → df_D01_SA01_R01 (19999 filas, columnas: ['ADXL', 'ITG', 'MMA'])


In [45]:
df_F01_SA01_R01_mag = importar_y_limpiar("F01_SA01_R01", magnitud=True)  # Cambiar a False si no se desea la magnitud
graficar_serie_tiempo(df_F01_SA01_R01_mag, 'ADXL', actividad="Caída Frontal por resbalón al caminar")

✔ 'F01_SA01_R01.txt' cargado → df_F01_SA01_R01 (3000 filas, columnas: ['ADXL', 'ITG', 'MMA'])


## Graficar en Frecuencia

##### Función para Graficar en Frecuencia

In [46]:
import numpy as np
import plotly.graph_objects as go

FS = 200  # Frecuencia de muestreo (Hz)


def _nombre_df(df, indice):
    """Busca el nombre de la variable (df_...) que apunta a este DataFrame."""
    for nombre, valor in globals().items():
        if valor is df and nombre.startswith("df_"):
            return nombre
    return f"df_{indice + 1}"  # respaldo si no se encuentra


def graficar_serie_frecuencia(datasets, columna, leyendas=None, fs=FS,
                              quitar_dc=True, fmax=None, devolver=False):
    """
    Grafica magnitud vs frecuencia (FFT) de una columna para uno o varios DataFrames.
    - datasets: lista de DataFrames, ej. [df_D01_SA01_R01, df_F06_SA03_R04]
                (también acepta un dict {nombre: df})
    - columna: 'ADXL', 'ITG', 'MMA' (o 'ADXL_X', etc. si importaste con magnitud=False)
    - leyendas: dict opcional {"df_D01_SA01_R01": "Trotando suavemente", ...}
    - fs: frecuencia de muestreo en Hz (por defecto 200)
    - quitar_dc: True resta la media antes de la FFT (elimina el pico en 0 Hz,
                 que en acelerómetros es la gravedad)
    - fmax: frecuencia máxima a mostrar en el eje X (Hz); None = hasta Nyquist (fs/2)
    """
    # Normalizar entrada a lista de (nombre, df)
    if isinstance(datasets, dict):
        items = list(datasets.items())
    else:
        items = [(_nombre_df(df, i), df) for i, df in enumerate(datasets)]

    leyendas = leyendas or {}
    unidad_y = "Magnitud (°/s)" if columna.startswith("ITG") else "Magnitud (g)"

    fig = go.Figure()
    for nombre, df in items:
        if columna not in df.columns:
            raise ValueError(f"'{columna}' no existe en {nombre}. "
                             f"Columnas disponibles: {list(df.columns)}")

        senal = df[columna].to_numpy(dtype=float)
        if quitar_dc:
            senal = senal - senal.mean()

        n = len(senal)
        espectro = np.abs(np.fft.rfft(senal)) * 2 / n   # amplitud normalizada
        frecuencias = np.fft.rfftfreq(n, d=1 / fs)

        fig.add_trace(go.Scatter(
            x=frecuencias,
            y=espectro,
            mode='lines',
            name=leyendas.get(nombre, nombre),
            line=dict(width=2)
        ))

    fig.update_layout(
        title=f"Espectro de Frecuencia: Sensor {columna}",
        xaxis_title="Frecuencia (Hz)",
        yaxis_title=unidad_y,
        template="plotly_white",
        hovermode="x unified",
        xaxis=dict(range=[0, fmax]) if fmax else {}
    )

    fig.show()
    if devolver:
        return fig

##### Persona caminando lento VS Resbalón Caminando lento (En Frecuencia)

In [47]:
df_D01_SA01_R01 = importar_y_limpiar("D01_SA01_R01", magnitud=True)
df_F01_SA01_R04 = importar_y_limpiar("F01_SA01_R04", magnitud=True)


dicc = {
    "df_D01_SA01_R01": "Caminando lento",
    "df_F01_SA01_R04": "Caída adelante caminando lento - Reslbalón"
}

graficar_serie_frecuencia([df_D01_SA01_R01, df_F01_SA01_R04], 'ADXL', leyendas=dicc)

✔ 'D01_SA01_R01.txt' cargado → df_D01_SA01_R01 (19999 filas, columnas: ['ADXL', 'ITG', 'MMA'])
✔ 'F01_SA01_R04.txt' cargado → df_F01_SA01_R04 (3000 filas, columnas: ['ADXL', 'ITG', 'MMA'])


## Función de Filtrado

In [48]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal

FS = 200  # Frecuencia de muestreo (Hz)

_TIPOS = {
    'lpf': 'lowpass', 'lowpass': 'lowpass', 'pasabajas': 'lowpass',
    'hpf': 'highpass', 'highpass': 'highpass', 'pasaaltas': 'highpass',
    'bpf': 'bandpass', 'bandpass': 'bandpass', 'pasabanda': 'bandpass', 'pasabandas': 'bandpass',
    'bsf': 'bandstop', 'bandstop': 'bandstop', 'rechazabanda': 'bandstop', 'rechazabandas': 'bandstop',
}
_MODELOS = ('butter', 'cheby1', 'cheby2', 'bessel', 'ellip')
_FUNC_ORDEN = {'butter': signal.buttord, 'cheby1': signal.cheb1ord,
               'cheby2': signal.cheb2ord, 'ellip': signal.ellipord}


def _diseñar_filtro(btype, modelo, fc, orden, fpaso, frechazo, gpass, gstop, rp, rs, fs):
    """Devuelve (sos, orden, Wn) según se especifique fc+orden o fpaso+frechazo."""
    es_banda = btype in ('bandpass', 'bandstop')

    if fpaso is not None and frechazo is not None:
        # Modo especificaciones: el orden se calcula automáticamente
        if modelo == 'bessel':
            raise ValueError("Bessel no tiene método de orden mínimo. Usa fc y orden.")
        orden, Wn = _FUNC_ORDEN[modelo](fpaso, frechazo, gpass, gstop, fs=fs)
        rp, rs = gpass, gstop
    elif fc is not None:
        Wn = fc
    else:
        raise ValueError("Especifica 'fc' (con 'orden') o bien 'fpaso' y 'frechazo'.")

    Wn_arr = np.atleast_1d(Wn)
    if es_banda and len(Wn_arr) != 2:
        raise ValueError("Para pasabandas/rechazabandas se necesitan dos frecuencias, ej. fc=(0.5, 10).")
    if not es_banda and len(Wn_arr) != 1:
        raise ValueError("Para LPF/HPF se necesita una sola frecuencia de corte.")
    if np.any(Wn_arr <= 0) or np.any(Wn_arr >= fs / 2):
        raise ValueError(f"Las frecuencias deben estar entre 0 y Nyquist ({fs/2} Hz).")

    kw = dict(btype=btype, output='sos', fs=fs)
    if modelo == 'butter':
        sos = signal.butter(orden, Wn, **kw)
    elif modelo == 'cheby1':
        sos = signal.cheby1(orden, rp, Wn, **kw)
    elif modelo == 'cheby2':
        sos = signal.cheby2(orden, rs, Wn, **kw)
    elif modelo == 'bessel':
        sos = signal.bessel(orden, Wn, norm='mag', **kw)  # 'mag': Wn corresponde a -3 dB
    elif modelo == 'ellip':
        sos = signal.ellip(orden, rp, rs, Wn, **kw)
    return sos, orden, Wn_arr


def filtrar_senal(df, columna, tipo='LPF', modelo='butter', fc=None, orden=4,
                  fpaso=None, frechazo=None, gpass=3, gstop=40, rp=1, rs=40,
                  fs=FS, causal=False, mostrar_bode=True, mostrar_senal=True):
    """
    Filtra una columna del DataFrame y muestra el diagrama de Bode del filtro.

    Parámetros
    - df, columna: DataFrame y columna a filtrar ('ADXL', 'ITG', 'MMA', ...)
    - tipo: 'LPF', 'HPF', 'BPF' (pasabandas) o 'BSF' (rechazabandas)
    - modelo: 'butter', 'cheby1', 'cheby2', 'bessel', 'ellip'
    - fc: frecuencia de corte en Hz (número para LPF/HPF, tupla (f1, f2) para bandas)
    - orden: orden del filtro (se usa con fc)
    - fpaso, frechazo: alternativa a fc+orden. Frecuencias de borde de la banda de paso y de
                       rechazo (tuplas en bandas). El orden mínimo se calcula solo.
                       Ej. LPF: fpaso=5, frechazo=10
                       Ej. BPF: fpaso=(1, 10), frechazo=(0.5, 15)
    - gpass, gstop: atenuación máxima en la banda de paso y mínima en la de rechazo (dB)
                    (solo en modo fpaso/frechazo)
    - rp, rs: rizado en banda de paso (cheby1, ellip) y atenuación en rechazo (cheby2, ellip) en dB
              (solo en modo fc+orden)
    - causal: False -> sosfiltfilt (offline, fase cero). True -> sosfilt (unidireccional, con retraso)
    - mostrar_bode / mostrar_senal: activa o desactiva cada gráfica

    Devuelve una pd.Series con la señal filtrada (nombre '<columna>_filt').
    """
    # --- Validaciones ---
    if columna not in df.columns:
        raise ValueError(f"'{columna}' no existe. Columnas disponibles: {list(df.columns)}")
    clave = tipo.lower().replace(' ', '').replace('-', '')
    if clave not in _TIPOS:
        raise ValueError(f"Tipo '{tipo}' no válido. Usa LPF, HPF, BPF o BSF.")
    btype = _TIPOS[clave]
    modelo = modelo.lower()
    if modelo not in _MODELOS:
        raise ValueError(f"Modelo '{modelo}' no válido. Opciones: {_MODELOS}")

    # --- Diseño (en forma SOS: numéricamente estable incluso con órdenes altos) ---
    sos, orden, Wn = _diseñar_filtro(btype, modelo, fc, orden, fpaso, frechazo,
                                     gpass, gstop, rp, rs, fs)

    # --- Aplicación del filtro ---
    x = df[columna].to_numpy(dtype=float)
    y = signal.sosfilt(sos, x) if causal else signal.sosfiltfilt(sos, x)
    filtrada = pd.Series(y, index=df.index, name=f"{columna}_filt")

    descripcion = (f"{modelo.capitalize()} {btype}, orden {orden}, "
                   f"Wn = {', '.join(f'{w:.3g}' for w in Wn)} Hz, "
                   f"{'causal (sosfilt)' if causal else 'offline (sosfiltfilt)'}")

    # --- Diagrama de Bode ---
    if mostrar_bode:
        w, h = signal.sosfreqz(sos, worN=8192, fs=fs)
        w, h = w[1:], h[1:]                       # quitar 0 Hz (eje logarítmico)
        mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-10))
        fase = np.degrees(np.unwrap(np.angle(h)))

        bode = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                             subplot_titles=("Magnitud", "Fase"))
        bode.add_trace(go.Scatter(x=w, y=mag_db, mode='lines', name='Un paso (freqz)',
                                  line=dict(width=2, color='royalblue')), row=1, col=1)
        if not causal:
            bode.add_trace(go.Scatter(x=w, y=2 * mag_db, mode='lines',
                                      name='Efectiva (ida y vuelta, filtfilt)',
                                      line=dict(width=2, color='firebrick', dash='dash')),
                           row=1, col=1)
        bode.add_trace(go.Scatter(x=w, y=fase, mode='lines', name='Fase (un paso)',
                                  line=dict(width=2, color='royalblue'), showlegend=False),
                       row=2, col=1)
        if not causal:
            bode.add_trace(go.Scatter(x=w, y=np.zeros_like(w), mode='lines',
                                      name='Fase efectiva = 0°',
                                      line=dict(width=2, color='firebrick', dash='dash'),
                                      showlegend=False), row=2, col=1)

        bode.add_hline(y=-3, line=dict(color='gray', dash='dot'), row=1, col=1,
                       annotation_text="-3 dB", annotation_position="bottom right")
        for f in Wn:
            bode.add_vline(x=f, line=dict(color='gray', dash='dot'))

        bode.update_xaxes(type='log', title_text="Frecuencia (Hz)", row=2, col=1)
        bode.update_xaxes(type='log', row=1, col=1)
        bode.update_yaxes(title_text="Magnitud (dB)", range=[-100, 5], row=1, col=1)
        bode.update_yaxes(title_text="Fase (°)", row=2, col=1)
        bode.update_layout(title=f"Diagrama de Bode: {descripcion}",
                           template="plotly_white", hovermode="x unified", height=650)
        bode.show()

    # --- Señal original vs filtrada ---
    if mostrar_senal:
        t = np.arange(len(x)) / fs
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=t, y=x, mode='lines', name='Original',
                                 line=dict(width=1.5, color='lightgray')))
        fig.add_trace(go.Scatter(x=t, y=y, mode='lines', name='Filtrada',
                                 line=dict(width=2, color='royalblue')))
        unidad = "Velocidad Angular (°/s)" if columna.startswith("ITG") else "Aceleración (g)"
        fig.update_layout(title=f"Señal {columna} filtrada: {descripcion}",
                          xaxis_title="Tiempo (Segundos)", yaxis_title=unidad,
                          template="plotly_white", hovermode="x unified")
        fig.show()

    return filtrada

##### Ahora probemos la función

In [51]:
df_D01_SA01_R01 = importar_y_limpiar("D01_SA01_R01", magnitud=True)

✔ 'D01_SA01_R01.txt' cargado → df_D01_SA01_R01 (19999 filas, columnas: ['ADXL', 'ITG', 'MMA'])


In [52]:
# LPF Butterworth a 5 Hz, orden 4, offline (fase cero)
df_D01_SA01_R01['ADXL_filt'] = filtrar_senal(df_D01_SA01_R01, 'ADXL', tipo='LPF', fc=5)

# # HPF Bessel a 0.5 Hz, causal (como en tiempo real)
# filtrar_senal(df_D01_SA01_R01, 'MMA', tipo='HPF', modelo='bessel', fc=0.5, orden=3, causal=True)

# # Pasabandas Chebyshev I, 1-10 Hz, 1 dB de rizado
# filtrar_senal(df_D01_SA01_R01, 'ITG', tipo='BPF', modelo='cheby1', fc=(1, 10), orden=4, rp=1)

# # Por especificaciones: el orden mínimo se calcula solo
# filtrar_senal(df_D01_SA01_R01, 'ADXL', tipo='LPF', modelo='ellip',
#               fpaso=5, frechazo=8, gpass=1, gstop=40)